In [13]:
import os
import numpy as np
import h5py
from tqdm import tqdm

In [14]:
ROOT = "/system/user/publicdata/gyrokinetics/raw"

In [15]:
def K_files(directory):
    files = os.listdir(directory)
    digit_files = sorted(
        [file for file in files if file.isdigit()], key=lambda x: int(x)
    )
    k_files = sorted(
        [file for file in files if file.startswith("K") and not file.endswith(".dat")]
    )
    return k_files + digit_files


def poten_files(directory):
    files = os.listdir(directory)
    poten_files = sorted([file for file in files if file.startswith("Poten")])
    timestep_slices = [int(f.replace("Poten", "")) for f in poten_files]
    return poten_files, np.array(timestep_slices) - 1

In [16]:
def preprocess(filename):
    dir_in = os.path.join(ROOT, filename)
    dir_out = "/system/user/publicdata/gyrokinetics/preprocessed"
    if not os.path.exists(dir_out):
        os.makedirs(dir_out)

    ks = K_files(dir_in)
    potens, ts_slices = poten_files(dir_in)
    # get timestamps
    ts = []
    for k in ks:
        # load corresponding timestep
        with open(os.path.join(dir_in, k + ".dat"), "r") as file:
            for line in file:
                line_split = line.split("=")
                if line_split[0].strip() == "TIME":
                    time = float(line_split[1].strip().strip(",").strip())
                    ts.append(time)
    timesteps = np.array(ts)

    # read helper vars
    time = np.loadtxt(os.path.join(dir_in, "time.dat"))
    sgrid = np.loadtxt(os.path.join(dir_in, "sgrid"))
    xphi = np.loadtxt(os.path.join(dir_in, "xphi"))
    yphi = np.loadtxt(os.path.join(dir_in, "yphi"))
    fluxes = np.loadtxt(os.path.join(dir_in, "fluxes.dat"))
    kyspec = np.loadtxt(os.path.join(dir_in, "kyspec"))
    krho = np.loadtxt(os.path.join(dir_in, "krho"))
    vpgr = np.loadtxt(os.path.join(dir_in, "vpgr.dat"))
    vperp = np.loadtxt(os.path.join(dir_in, "vperp.dat"))
    # number of parallel direction grid points
    ns = sgrid.shape[1] if len(sgrid.shape) > 1 else sgrid.shape[0]
    # number of x, y grid points (in real space)
    nx, ny = xphi.shape[1], xphi.shape[0]
    # number of modes in x and y direction
    nkx, nky = krho.shape[1], krho.shape[0]
    # get velocity space resolutions
    nvpar, nmu = vpgr.shape[1], vpgr.shape[0]

    # load fluxes
    fluxes = np.loadtxt(os.path.join(dir_in, "fluxes.dat"))[:, 1]
    fluxes = fluxes[ts_slices]

    # create h5 file with timestamps and field data
    h5_filename = os.path.join(dir_out, filename + ".h5")
    with h5py.File(h5_filename, "w") as file:
        # group for metadata (e.g. timesteps)
        metadata_group = file.create_group("metadata")
        metadata_group.create_dataset("timesteps", data=timesteps)
        metadata_group.create_dataset("fluxes", data=fluxes)

        # group for our 6D field data
        data_group = file.create_group("data")
        for idx, (k, pot) in tqdm(
            enumerate(zip(ks, potens)), f"Processing {filename}", total=len(ks)
        ):

            # Load the full distribution function data
            with open(os.path.join(dir_in, k), "rb") as fid:
                ff = np.fromfile(fid, dtype=np.float64)

            # Reshape the distribution function
            knth = np.reshape(ff, (2, nvpar, nmu, ns, nkx, nky), order="F").astype(
                "float32"
            )

            # Add the reshaped data as a dataset to the "data" group
            k_name = "timestep_" + str(idx).zfill(5)
            data_group.create_dataset(k_name, data=knth)

            # load the potential field
            a = np.loadtxt(os.path.join(dir_in, pot))
            phi = np.reshape(a, (nx, ns, ny)).astype("float32")
            poten_name = "poten_" + str(idx).zfill(5)
            data_group.create_dataset(poten_name, data=phi)

            # print(f"Added dataset '{k_name}'+'{poten_name}' to HDF5 file.")
        return h5_filename

In [ ]:
datasets = ["cyclone4_2", "cyclone5_2", "cyclone6_2", "cyclone7_2", "cyclone8_2"]

for f in datasets:
    h5_filename = preprocess(f)
    # set rwx permissions
    try:
        os.chmod(h5_filename, 0o777)
    except PermissionError:
        pass
    # read in the structure and example field of the created h5 file
    with h5py.File(h5_filename, "r") as h5f:
        # Read the 'metadata/timesteps' dataset
        timesteps = h5f["metadata/timesteps"][:]
        timestep_0 = h5f["data/timestep_00000"][:]
        print(
            f"{h5_filename}, timesteps: {len(timesteps)}, shape of timestep_00000: {timestep_0.shape}"
        )